# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

# My summary what we are going to build Day 5 is the capstone project for Week 1 — it builds a full company brochure generator. Here's what's in it:

The project: Give it a company name + URL → it produces a polished markdown brochure for clients, investors, and recruits.

What's new vs Day 1:

Multi-step LLM pipeline — first LLM call decides which links on a page are relevant (about, careers, etc.) and returns structured JSON. Second LLM call reads all that content and writes the brochure. Two LLMs working in sequence — your first taste of agentic design.

JSON mode — response_format={"type": "json_object"} forces the model to respond in valid JSON. No parsing hacks, guaranteed structure. Used for the link-filtering step.

One-shot prompting — the system prompt includes an example of the expected JSON output. Model follows the pattern. Simple but effective technique.

Streaming — stream=True returns the response token by token, rendered live with the typewriter effect using update_display(). This is how ChatGPT's UI works.

Tone control — swapping the system prompt from professional to humorous completely changes the output. Same data, same pipeline, different persona. Demonstrates how much control the system prompt gives you.

The bigger point the course is making: This is the first example of chaining multiple LLM calls together — which is the foundation of agentic AI (Week 8). Day 1 was one call, Day 5 is a small pipeline.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [ ]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
openai = OpenAI(
   base_url="http://localhost:11434/v1",
   api_key="ollama"
)
MODEL="qwen2.5:7b"

There might be a problem with your API key? Please visit the troubleshooting notebook!


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are given a list of links from a company webpage.
Select only links useful for a company brochure (about, products, services, careers, press, blog posts, company sites).
Skip: anchor links (#...), mailto: links, duplicate URLs, social media, unrelated external sites.

Respond ONLY with valid JSON in this exact format - no other keys, no extra text:
{
    "links": [
        {"type": "about page", "url": "https://example.com/about", "reason": "describes the company"},
        {"type": "careers page", "url": "https://example.com/careers", "reason": "useful for recruits"},
        {"type": "blog post", "url": "https://example.com/post-1", "reason": "shows company expertise"}
    ]
}

Important:
- "links" is the only key allowed in your response
- Use full https:// URLs only
- Remove duplicate URLs
- Include all relevant links you find
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url}.
Select all relevant links for a company brochure and respond in JSON format.
Do not include Terms of Service, Privacy, email links, or anchor links.

Links (some might be relative - convert them to full URLs):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com.
Select all relevant links for a company brochure and respond in JSON format.
Do not include Terms of Service, Privacy, email links, or anchor links.

Links (some might be relative - convert them to full URLs):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder

In [8]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/',
   'reason': 'describes the company'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/posts/',
   'reason': 'shows company expertise'},
  {'type': 'careers page', 'url': '', 'reason': ''},
  {'type': 'products page', 'url': '', 'reason': ''},
  {'type': 'services page', 'url': '', 'reason': ''},
  {'type': 'press page',
   'url': 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
   'reason': 'mentions acquisitions'},
  {'type': 'company site',
   'url': 'https://edwarddonner.com/',
   'reason': 'official website'}]}

In [10]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [12]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2:latest
Found 5 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/',
   'reason': 'describes the company'},
  {'type': 'posts',
   'url': 'https://edwarddonner.com/posts/',
   'reason': 'latest blog posts from Edward Donner'},
  {'type': 'press',
   'url': 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
   'reason': 'news about the company'},
  {'type': 'careers page',
   'url': 'https://edwarddonner.com/curriculum/',
   'reason': 'useful for recruits'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
   'reason': 'shows company expertise'}]}

In [13]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:latest
Found 7 relevant links


{'links': [{'type': 'about the company', 'url': 'https://huggingface.co/'},
  {'type': 'blog post', 'url': 'https://docs.huggingface.co/blog.html'},
  {'type': 'news posts', 'url': 'https://status.huggingface.co/'},
  {'type': 'github repository', 'url': 'https://github.com/huggingface'},
  {'type': 'hugging face discuss forum',
   'url': 'https://discuss.huggingface.co/'},
  {'type': 'social media twitter handle',
   'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin company page',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        try:
            result += f"\n\n### Link: {link['type']}\n"
            result += fetch_website_contents(link["url"])
        except Exception as e:
            print(f"Skipping {link['url']} — {e}")
    return result

In [18]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling llama3.2:latest
Found 8 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
nvidia/LocateAnything-3B
Updated
10 days ago
•
111k
•
1.42k
google/gemma-4-12B-it
Updated
2 days ago
•
315k
•
592
unsloth/gemma-4-12b-it-GGUF
Updated
1 day ago
•
458k
•
401
google/gemma-4-12B
Updated
2 days ago
•
84.5k
•
360
HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive
Updated
A

In [13]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [14]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:latest
Found 9 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nnvidia/LocateAnything-3B\nUpdated\n10 days ago\n•\n111k\n•\n1.42k\ngoogle/gemma-4-12B-it\nUpdated\n2 days ago\n•\n315k\n•\n595

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [19]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:latest
Found 9 relevant links
Skipping mailto:press@huggingface.co — No connection adapters were found for 'mailto:press@huggingface.co'


Hugging Face Brochure
=====================

** Mission to Empower AI Community**

Hugging Face is a collaborative platform connecting machine learning practitioners, researchers, and developers. Our mission is to foster an open, inclusive, and ethical AI community by providing the tools, resources, and infrastructure for creating advanced AI models.

**Our Product Stack**
-------------------

*   **Models**: Browse 2+ million pre-trained models for various tasks, such as image classification, object detection, and natural language processing.
*   **Datasets**: Discover and access over 500,000 open-source datasets for machine learning development.
*   **Spaces**: Develop, deploy, and share AI applications with our zero-code platform powered by Agents.
*   **Inference Endpoints**: Leverage our scalable inference service to deploy models in production.

**Company Culture**
------------------

At Hugging Face, we value collaboration, trust, and respect. Our diverse team of experts fosters an inclusive environment that encourages peer feedback, mentorship, and shared learning.

*   We're passionate about open-source contributions.
*   Community-driven discussions and GitHub PR review ensure high-quality code changes.
*   Collaboration with research institutions and organizations pushes the boundary of AI advancements.

**Customer Base**
----------------

Companies, researchers, and developers from diverse backgrounds work with Hugging Face to accelerate their AI transformations.

*   **Research**: Partnering with top-tier academia for cutting-edge projects in NLP, computer vision, and other areas.
*   **Enterprise**: Empowering large-scale deployments through customizable solutions and dedicated support.

**Careers & Opportunities**
------------------------

Join our dynamic team of engineers, researchers, and software developers. Explore job openings and stay connected on social media for the latest updates.

*   [Discover our Careers page](link)
*   Follow Hugging Face on LinkedIn to unlock new opportunities.
*   Build your professional profile by showcasing your contributions to open-source projects.

**Why Choose Hugging Face?**
-----------------------------

Participate in the AI community that drives AI's future. Our commitment to collaboration, ethics, and innovation creates a platform where diverse voices and talents are valued.

*   Expand your technical expertise through hands-on experience.
*   Align yourself with progressive AI values.
*   Contribute directly to shaping an open-source ecosystem for groundbreaking machine learning applications.

Stay ahead of the curve!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [20]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [21]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:latest
Found 10 relevant links
Skipping https://blog.huggingface.co/ — HTTPSConnectionPool(host='blog.huggingface.co', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000002475BAE7650>: Failed to resolve 'blog.huggingface.co' ([Errno 11001] getaddrinfo failed)"))
Skipping https://changelog.huggingface.co/ — HTTPSConnectionPool(host='changelog.huggingface.co', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000002475B64FDA0>: Failed to resolve 'changelog.huggingface.co' ([Errno 11001] getaddrinfo failed)"))
Skipping https://learn.huggingface.co/ — HTTPSConnectionPool(host='learn.huggingface.co', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000002475BA2C2F0>: Failed to resolve 'learn.huggin

# Hugging Face: Empowering the Future of Machine Learning

Welcome to Hugging Face, a pioneering platform that's redefining the intersection of machine learning and AI. As a leading innovator in this space, we're dedicated to building a community that fosters collaboration, discovery, and progress.

## About Us
At Hugging Face, we believe that machine learning should be accessible, user-friendly, and fun. Our platform provides a vast repository of pre-trained models, datasets, and applications, empowering developers and researchers to accelerate their projects and achieve remarkable results.

## Collaboration and Community

Join our vibrant community of over 2 million users who come together to share knowledge, showcase projects, and contribute to the growth of Machine Learning through our:

*   **Hugging Face Forum**: Engage with experts, ask questions, and receive high-quality feedback from our active community.
*   **Discord Channel**: Participate in real-time discussions, share updates, and get involved in initiatives shaping the future of AI.

## Product Offerings
Our diverse range of products cater to different needs and applications:

*   **Hugging Face Models**: Browse over 2 million pre-trained models for various tasks, including text, image, and audio processing.
*   **Datasets**: Tap into our extensive collection of datasets, from text to image, tabular data, and more.
*   **Spaces and Buckets**: Efficiently store, manage, and deploy your models with our zero-cost infrastructure.

## Enterprise Solutions
Discover how our platform can help you drive business growth:

*   **Hugging Face PRO**: Unlock the full potential of our platform for personalized support, dedicated onboarding, and priority access to new releases.
*   **Enterprise Support**: Expertise at scale – from custom solutions to strategic consulting, we're here to guide your team.

## The Benefits
Why choose Hugging Face?

*   **Ease of Use**: Our intuitive interface ensures that you can focus on building your project, not struggling with the technology.

    At  HuggingFace, our mission is to accelerate the world's transition to trustworthy AI. Join us in shaping the future of machine learning and empower yourself, your team, or your organization to unlock new possibilities!

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>